In [16]:
import pandas as pd
import geopandas as gpd
import fiona
import os

## Uncomment when running manually ##
# Read the unique values dictionary from the Excel file
# unique_values_mine_gdf = pd.read_excel("min_processing_values_dict_New.xlsx")
# #result_dict = {(row['mineral'], row['stage_cut']): row['processing_energy_intensity'] for index, row in unique_values_mine_gdf.iterrows()}
# scnio_name = 'precursor_2040_mid_max_threshold_metal_tons_region_unconstrained'
# scnio = f"Outputs/Results/{scnio_name}/{scnio_name}.csv"
# scope = 'region' # choose between region and country
# output_path = 'Outputs/Final_Output_per_scenario'
# file = os.path.join(output_path, f'{scnio_name}.csv')

## Use the code below when running with snakemake ##
unique_values_mine_gdf = pd.read_excel(snakemake.input.min_processing_values)
scnio = snakemake.input.scenario
scope = snakemake.params.scope
output_path = snakemake.params.output_path
file = snakemake.output.result
###############
result_dict = {(row['isocc'], row['mineral'], row['stage_cut']): row['processing_energy_intensity'] for index, row in unique_values_mine_gdf.iterrows()}

result = pd.read_csv(scnio)

In [ ]:
def calculate_demand_country(row):
    total_demand = 0
    demand_by_combination = {}
    country_code = row['iso3']  # or row['country'] if you use full name
    
    for (c, mineral, stage_cut), energy_intensity in result_dict.items():
        if c != country_code:
            continue
        production_col = f"{mineral}_production_tons_{stage_cut}_in_country"
        if production_col in row:
            try:
                production_amount = float(row[production_col])
                demand = (production_amount * 1000) * energy_intensity
                total_demand += demand
                demand_by_combination[f"demand_{mineral}_{stage_cut}"] = demand
            except (ValueError, TypeError):
                print(f"Warning: Non-numeric value found in column {production_col}")
                demand_by_combination[f"demand_{mineral}_{stage_cut}"] = 0
                continue
    
    demand_by_combination["total_demand_postan"] = total_demand
    return demand_by_combination

In [ ]:
def calculate_demand_region(row):
    total_demand = 0
    demand_by_combination = {}
    country_code = row['iso3']  # or row['country'] if you use full name
    
    for (c, mineral, stage_cut), energy_intensity in result_dict.items():
        if c != country_code:
            continue
        production_col = f"{mineral}_production_tons_{stage_cut}_in_region"
        if production_col in row:
            try:
                production_amount = float(row[production_col])
                demand = (production_amount * 1000) * energy_intensity
                total_demand += demand
                demand_by_combination[f"demand_{mineral}_{stage_cut}"] = demand
            except (ValueError, TypeError):
                print(f"Warning: Non-numeric value found in column {production_col}")
                demand_by_combination[f"demand_{mineral}_{stage_cut}"] = 0
                continue
    
    demand_by_combination["total_demand_postan"] = total_demand
    return demand_by_combination

In [19]:
############# RUN  #######################
if scope == 'country':
    demand_results = result.apply(calculate_demand_country, axis=1)
    demand_results_df = pd.DataFrame(demand_results.tolist())
else:
    demand_results = result.apply(calculate_demand_region, axis=1)
    demand_results_df = pd.DataFrame(demand_results.tolist())

In [20]:
# Add the new columns to the mine_gdf
result = pd.concat([result, demand_results_df], axis=1)

In [21]:
unique_values_mine_gdf.mineral.unique()

# Initialize an empty dictionary to store the summed values for each mineral
mineral_sums = {}

# Iterate over each mineral and sum the relevant columns
for mineral in unique_values_mine_gdf.mineral.unique():
    # Filter columns that contain the mineral name
    mineral_columns = [col for col in demand_results_df.columns if mineral in col]
    #print (mineral_columns)
    # Sum the values of these columns
    mineral_sums[mineral] = result[mineral_columns].sum(axis=1)

# Create new columns in the dataframe for each mineral
for mineral, sums in mineral_sums.items():
    result[f'{mineral}_demand_sum'] = sums

In [ ]:
def calculate_demandper_country(row):
    total_demandper = 0
    demandper_by_combination = {}
    country_code = row['iso3']  # or row['country'] if you use full name
    
    for (c, mineral, stage_cut), energy_intensity in result_dict.items():
        if c != country_code:
            continue
        production_col = f"{mineral}_production_tons_{stage_cut}_in_country"
        if production_col in row:
            try:
                production_amount = float(row[production_col])
                demand = (production_amount * 1000) * energy_intensity
                demandper = demand / (float(row["demand_country"])+0.00000000000001)
                total_demandper += demandper
                #demand_by_combination[f"demand_{mineral}_{stage_cut}"] = demand
                demandper_by_combination[f"demandper_{mineral}_{stage_cut}"] = demandper
            except (ValueError, TypeError):
                print(f"Warning: Non-numeric value found in column {production_col}")
                #demand_by_combination[f"demand_{mineral}_{stage_cut}"] = 0
                demandper_by_combination[f"demandper_{mineral}_{stage_cut}"] = 0                
                continue
    
    demandper_by_combination["total_demandper_postan"] = total_demandper
    return demandper_by_combination

In [ ]:
def calculate_demandper_region(row):
    total_demandper = 0
    demandper_by_combination = {}
    country_code = row['iso3']  # or row['country'] if you use full name
    
    for (c, mineral, stage_cut), energy_intensity in result_dict.items():
        if c != country_code:
            continue
        production_col = f"{mineral}_production_tons_{stage_cut}_in_region"
        if production_col in row:
            try:
                production_amount = float(row[production_col])
                demand = (production_amount * 1000) * energy_intensity
                demandper = demand / (float(row["demand_region"])+0.00000000000001)
                total_demandper += demandper
                #demand_by_combination[f"demand_{mineral}_{stage_cut}"] = demand
                demandper_by_combination[f"demandper_{mineral}_{stage_cut}"] = demandper
            except (ValueError, TypeError):
                print(f"Warning: Non-numeric value found in column {production_col}")
                #demand_by_combination[f"demand_{mineral}_{stage_cut}"] = 0
                demandper_by_combination[f"demandper_{mineral}_{stage_cut}"] = 0                
                continue
    
    demandper_by_combination["total_demandper_postan"] = total_demandper
    return demandper_by_combination

In [24]:
############# RUN  #######################
if scope == 'country':
    demandper_results = result.apply(calculate_demandper_country, axis=1)
    demandper_results_df = pd.DataFrame(demandper_results.tolist())
else:
    demandper_results = result.apply(calculate_demandper_region, axis=1)
    demandper_results_df = pd.DataFrame(demandper_results.tolist())

In [25]:
# Add the new columns to the mine_gdf
result = pd.concat([result, demandper_results_df], axis=1)

In [26]:
############# RUN  #######################

unique_values_mine_gdf.mineral.unique()

if scope == 'country':
## Iterate over each mineral and sum the relevant columns
    for mineral in unique_values_mine_gdf.mineral.unique():
       # Filter columns that contain the mineral name
       result[f'{mineral}_per'] = result[f'{mineral}_demand_sum'] / result['demand_country']
else:
# Iterate over each mineral and sum the relevant columns
    for mineral in unique_values_mine_gdf.mineral.unique():
        # Filter columns that contain the mineral name
        result[f'{mineral}_per'] = result[f'{mineral}_demand_sum'] / result['demand_region']

In [27]:
unique_values_mine_gdf.mineral.unique()

# Iterate over each mineral and sum the relevant columns
for mineral in unique_values_mine_gdf.mineral.unique():
    # Filter columns that contain the mineral name
    result[f'{mineral}_INV_USD'] = result[f'{mineral}_per'] * result['Final_Investment_USD']
    result[f'{mineral}_CAP_kW'] = result[f'{mineral}_per'] * result['Req_Capacity_kW']
    result[f'{mineral}_EMI_tCO2eq'] = result[f'{mineral}_per'] * result['Emissions_tCO2eq']

In [28]:
#result.to_csv("zanzizanzi.csv")

In [29]:
os.makedirs(output_path, exist_ok=True)
result.to_csv(file, index=False)

In [30]:
##Identify relevant columns
#relevant_keywords = ["cobalt", "lithium", "copper", "nickel", "manganese", "graphite"]
#relevant_columns = [col for col in result.columns if any(keyword in col for keyword in relevant_keywords)]
#unique_final_stage_columns = [col for col in relevant_columns if 'INV_USD' in col]
#
## Filter the DataFrame for the specified columns
#filtered_df = result[unique_final_stage_columns + ['Final_Investment_USD']]
#total_inv_sum = filtered_df.sum()

In [31]:
##Identify relevant columns
#relevant_keywords = ["cobalt", "lithium", "copper", "nickel", "manganese", "graphite"]
#relevant_columns = [col for col in result.columns if any(keyword in col for keyword in relevant_keywords)]
#unique_final_stage_columns = [col for col in relevant_columns if 'CAP_kW' in col]
#
## Filter the DataFrame for the specified columns
#filtered_df = result[unique_final_stage_columns + ['Req_Capacity_kW']]
#total_cap_sum = filtered_df.sum()

In [32]:
##Identify relevant columns
#relevant_keywords = ["cobalt", "lithium", "copper", "nickel", "manganese", "graphite"]
#relevant_columns = [col for col in result.columns if any(keyword in col for keyword in relevant_keywords)]
#unique_final_stage_columns = [col for col in relevant_columns if 'EMI_tCO2eq' in col]
#
## Filter the DataFrame for the specified columns
#filtered_df = result[unique_final_stage_columns + ['Emissions_tCO2eq']]
#total_emi_sum = filtered_df.sum()

In [33]:
#total_inv_sum, total_cap_sum, total_emi_sum